In [1]:
# Install LangGraph for building stateful graph-based agent workflows.

%pip install -U langgraph

  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.10
    Uninstalling langgraph-1.2.10:
      Successfully uninstalled langgraph-1.2.10
Note: you may need to restart the kernel to use updated packages.


In [20]:
# LangGraph core concepts — explained before we build anything.
summary = """
STATEGRAPH:
The builder object. You use it to add nodes and edges, then compile it.

STATE:
Shared data that moves through the whole graph. Every node can read
and update it.

NODE:
A function that does one step (search, draft, critique...) and
returns updates to the state.

EDGE:
A fixed path: after node A, always go to node B.

CONDITIONAL EDGE:
A smart path: a function checks the state and decides where to go next.

CYCLE:
A loop back to an earlier node, so the graph can retry/improve instead
of just moving forward once.

COMPILE:
Turns the graph design into something you can actually run.
"""
print(summary)


STATEGRAPH:
The builder object. You use it to add nodes and edges, then compile it.

STATE:
Shared data that moves through the whole graph. Every node can read
and update it.

NODE:
A function that does one step (search, draft, critique...) and
returns updates to the state.

EDGE:
A fixed path: after node A, always go to node B.

CONDITIONAL EDGE:
A smart path: a function checks the state and decides where to go next.

CYCLE:
A loop back to an earlier node, so the graph can retry/improve instead
of just moving forward once.

COMPILE:
Turns the graph design into something you can actually run.



                     ┌─────────────┐
                     │    START    │
                     └──────┬──────┘
                            │
                            ▼
                     ┌─────────────┐
                     │    SEARCH   │
                     └──────┬──────┘
                            │
                            ▼
                     ┌─────────────┐
                     │    DRAFT    │
                     └──────┬──────┘
                            │
                            ▼
                     ┌─────────────┐
                     │   CRITIQUE  │
                     └──────┬──────┘
                            │
                ┌───────────┴───────────┐
                │                       │
          score >= 0.8             score < 0.8
                │                       │
                ▼                       ▼
           ┌─────────┐            ┌──────────┐
           │   END   │            │  REVISE  │
           └─────────┘            └────┬─────┘
                                       │
                                       │
                                       └──────► CRITIQUE

In [2]:
# Import LangGraph and Python typing components needed to define the graph and its state.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List
print("LangGraph imported successfully.")

LangGraph imported successfully.


In [3]:
# Define the workflow stages that our research assistant will eventually follow.
workflow_description = """
Research Assistant Workflow
1. User provides a research question.
2. Search node gathers relevant information.
3. Draft node creates an initial answer.
4. Critique node evaluates the draft.
5. If the answer is weak, the graph returns to Draft.
6. If the answer is acceptable, the graph moves to Finish.
"""
print(workflow_description)


Research Assistant Workflow
1. User provides a research question.
2. Search node gathers relevant information.
3. Draft node creates an initial answer.
4. Critique node evaluates the draft.
5. If the answer is weak, the graph returns to Draft.
6. If the answer is acceptable, the graph moves to Finish.



In [4]:
# Define the shared state that will travel through every node of the graph.
class ResearchState(TypedDict):
    question: str
    search_results: List[str]
    draft: str
    critique: str
    quality_score: float
    revision_count: int

In [5]:
# Display the fields defined in the research assistant state.

print("ResearchState fields:")
for field_name, field_type in ResearchState.__annotations__.items():
    print(f"{field_name}: {field_type}")

ResearchState fields:
question: <class 'str'>
search_results: typing.List[str]
draft: <class 'str'>
critique: <class 'str'>
quality_score: <class 'float'>
revision_count: <class 'int'>


In [6]:
# Define placeholder node functions to demonstrate how each graph node reads and updates shared state.

def search_node(state: ResearchState):
    return {
        "search_results": [
            "Research result 1",
            "Research result 2"
        ]
    }
def draft_node(state: ResearchState):
    return {
        "draft": "Initial research answer based on the available information."
    }
def critique_node(state: ResearchState):
    return {
        "critique": "The answer is reasonable but needs more supporting evidence.",
        "quality_score": 0.7
    }
def revise_node(state: ResearchState):
    return {
        "draft": "Improved research answer with additional supporting evidence.",
        "revision_count": state["revision_count"] + 1
    }
print("Node functions defined successfully.")

Node functions defined successfully.


In [153]:
# Create a StateGraph whose shared state follows the ResearchState schema.
graph_builder = StateGraph(ResearchState)
print("StateGraph created successfully.")

StateGraph created successfully.


#### StateGraph is the object used to design the graph.
##### Think of it as the blueprint:

StateGraph
    │
    ├── Nodes
    ├── Edges
    ├── Conditional Edges
    └── State

In [154]:
# Register each workflow function as a named node in the LangGraph.
graph_builder.add_node("search", search_node)
graph_builder.add_node("draft", draft_node)
graph_builder.add_node("critique", critique_node)
graph_builder.add_node("revise", revise_node)
print("Nodes registered:")
print("search")
print("draft")
print("critique")
print("revise")

Nodes registered:
search
draft
critique
revise


In [ ]:
# Connect the search, draft, and critique nodes using normal directed edges.
graph_builder.add_edge(START, "search")
graph_builder.add_edge("search", "draft")
graph_builder.add_edge("draft", "critique")
print("Linear edges added successfully.")

Linear edges added successfully.


In [ ]:
# Define a routing function that chooses whether the graph should finish or revise the draft.
def critique_router(state: ResearchState):
    if state["quality_score"] >= 0.8:
        return "finish"
    else:
        return "revise"
print("Conditional router created.")

Conditional router created.


In [ ]:
# Add a conditional edge that routes the workflow based on the critique quality score.
graph_builder.add_conditional_edges(
    "critique",
    critique_router,
    {
        "finish": END,
        "revise": "revise"
    }
)
print("Conditional edge added successfully.")

Conditional edge added successfully.


In [ ]:
# Connect the revision node back to the critique node to create the self-correction cycle.
graph_builder.add_edge("revise", "critique")
print("Revision loop created successfully.")

Revision loop created successfully.


In [ ]:
# Generate a Mermaid representation of the graph so its structure can be visually inspected.
print("""
START
  |
  v
SEARCH
  |
  v
DRAFT
  |
  v
CRITIQUE
  |-------- quality >= 0.8 --------> END
  |
  -------- quality < 0.8 ---------> REVISE
                                      |
                                      v
                                   CRITIQUE
""")


START
  |
  v
SEARCH
  |
  v
DRAFT
  |
  v
CRITIQUE
  |-------- quality >= 0.8 --------> END
  |
  -------- quality < 0.8 ---------> REVISE
                                      |
                                      v
                                   CRITIQUE



In [ ]:
# Compile the graph blueprint into an executable LangGraph application.
graph = graph_builder.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


In [ ]:
# Verify that the compiled graph object is ready for execution.
print("Compiled graph type:", type(graph))
print("Graph is ready for Task 2 execution.")

Compiled graph type: <class 'langgraph.graph.state.CompiledStateGraph'>
Graph is ready for Task 2 execution.


In [ ]:
# Create the initial state that will enter the research assistant graph.
initial_state: ResearchState = {
    "question": "What are the benefits of artificial intelligence in healthcare?",
    "search_results": [],
    "draft": "",
    "critique": "",
    "quality_score": 0.0,
    "revision_count": 0
}
print("Initial state:")
print(initial_state)

Initial state:
{'question': 'What are the benefits of artificial intelligence in healthcare?', 'search_results': [], 'draft': '', 'critique': '', 'quality_score': 0.0, 'revision_count': 0}


```mermaid
flowchart TD
    START([START]) --> SEARCH[Search]
    SEARCH --> DRAFT[Draft]
    DRAFT --> CRITIQUE[Critique]
    CRITIQUE -->|quality_score >= 0.8| END([END])
    CRITIQUE -->|quality_score < 0.8| REVISE[Revise]
    REVISE --> CRITIQUE
```

In [ ]:
# Import the LangGraph components and typing tools needed for the linear workflow.
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
# Define a separate state schema for the product workflow so it does not overlap with Task 1.
class ProductWorkflowState(TypedDict):
    product_name: str
    plan: str
    retrieved_data: dict
    generated_answer: str
    formatted_answer: str

print("ProductWorkflowState created successfully.")

ProductWorkflowState created successfully.


In [1]:
# Recreate the Day 2 CSV-backed data source (same structure, same values).
import pandas as pd
products_data = {
    "product_id": [101, 102, 103, 104],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "price": [120000, 2500, 4500, 35000],
    "category": ["Electronics", "Accessories", "Accessories", "Electronics"]
}
products_df = pd.DataFrame(products_data)
CSV_PATH = "products.csv"
products_df.to_csv(CSV_PATH, index=False)

print(f"CSV created successfully: {CSV_PATH}")

CSV created successfully: products.csv


In [ ]:
# Exact same lookup_product tool from Day 2 — CSV-backed, not a hardcoded dict.
from langchain_core.tools import tool
@tool
def lookup_product(product_name: str) -> dict:
    """
    Search the local products CSV file for a product and return its
    product ID, name, price, and category.

    Use this tool when the user asks about the price, category,
    or details of a product.
    """
    try:
        df = pd.read_csv(CSV_PATH)
        matches = df[
            df["product"].str.lower() == product_name.strip().lower()
        ]
        if matches.empty:
            return {"error": f"Product '{product_name}' was not found."}
        product = matches.iloc[0]
        return {
            "product_id": int(product["product_id"]),
            "product": product["product"],
            "price": float(product["price"]),
            "category": product["category"]
        }
    except Exception as e:
        return {"error": str(e)}

print("Day 2 lookup_product tool (CSV-backed) reused successfully.")

Day 2 lookup_product tool (CSV-backed) reused successfully.


In [ ]:
# The plan node determines what information should be retrieved for the requested product.
def plan_product(state: ProductWorkflowState):
    product_name = state["product_name"]

    return {
        "plan": f"Retrieve the price and category information for {product_name}."
    }

print("Plan node created successfully.")

Plan node created successfully.


In [ ]:
# Retrieve node calls the real Day 2 CSV-backed tool.
def retrieve_product(state: ProductWorkflowState):
    product_name = state["product_name"]
    result = lookup_product.invoke({"product_name": product_name})
    return {
        "retrieved_data": result
    }

print("Retrieve node created successfully — calling Day 2's CSV-backed lookup_product tool.")

Retrieve node created successfully — calling Day 2's CSV-backed lookup_product tool.


In [ ]:
# Generate node — unchanged logic, but now product_id is available if you want to show it.
def generate_product_answer(state: ProductWorkflowState):
    data = state["retrieved_data"]
    if "error" in data:
        return {
            "generated_answer": data["error"]
        }
    answer = (
        f"The {data['product']} (ID: {data['product_id']}) costs {data['price']} PKR "
        f"and belongs to the {data['category']} category."
    )
    return {
        "generated_answer": answer
    }

print("Generate node created successfully.")

Generate node created successfully.


In [ ]:
# The format node converts the generated answer into the final presentation format.
def format_product_answer(state: ProductWorkflowState):
    return {
        "formatted_answer": f"FINAL ANSWER: {state['generated_answer']}"
    }
print("Format node created successfully.")

Format node created successfully.


In [ ]:
# Create a separate StateGraph for Task 2's linear product workflow.
product_graph_builder = StateGraph(ProductWorkflowState)
print("Task 2 StateGraph created successfully.")

Task 2 StateGraph created successfully.


In [ ]:
# Register the four linear workflow stages: plan, retrieve, generate, and format.
product_graph_builder.add_node("plan", plan_product)
product_graph_builder.add_node("retrieve", retrieve_product)
product_graph_builder.add_node("generate", generate_product_answer)
product_graph_builder.add_node("format", format_product_answer)

print("Task 2 nodes registered successfully.")

Task 2 nodes registered successfully.


In [ ]:
# Connect the four nodes sequentially to create a linear workflow.
product_graph_builder.add_edge(START, "plan")
product_graph_builder.add_edge("plan", "retrieve")
product_graph_builder.add_edge("retrieve", "generate")
product_graph_builder.add_edge("generate", "format")
product_graph_builder.add_edge("format", END)
print("Task 2 linear edges added successfully.")

Task 2 linear edges added successfully.


In [ ]:
# Compile the linear graph so it becomes executable.
product_graph = product_graph_builder.compile()
print("Task 2 graph compiled successfully.")

Task 2 graph compiled successfully.


In [ ]:
# Create the starting state for a sample product query.
product_initial_state: ProductWorkflowState = {
    "product_name": "Laptop",
    "plan": "",
    "retrieved_data": {},
    "generated_answer": "",
    "formatted_answer": ""
}

print("Initial state:")
print(product_initial_state)

Initial state:
{'product_name': 'Laptop', 'plan': '', 'retrieved_data': {}, 'generated_answer': '', 'formatted_answer': ''}


In [ ]:
# Stream the graph execution so we can inspect the state produced by every node.
print("=" * 70)
print("TASK 2 — LINEAR GRAPH EXECUTION")
print("=" * 70)
for event in product_graph.stream(product_initial_state):
    for node_name, node_state in event.items():
        print("\n" + "=" * 70)
        print("NODE EXECUTED:", node_name)
        print("=" * 70)
        print("STATE UPDATE:")
        print(node_state)

TASK 2 — LINEAR GRAPH EXECUTION

NODE EXECUTED: plan
STATE UPDATE:
{'plan': 'Retrieve the price and category information for Laptop.'}

NODE EXECUTED: retrieve
STATE UPDATE:
{'retrieved_data': {'product_id': 101, 'product': 'Laptop', 'price': 120000.0, 'category': 'Electronics'}}

NODE EXECUTED: generate
STATE UPDATE:
{'generated_answer': 'The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.'}

NODE EXECUTED: format
STATE UPDATE:
{'formatted_answer': 'FINAL ANSWER: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.'}


In [ ]:
# Invoke the complete graph once and display the final accumulated state.
final_product_state = product_graph.invoke(product_initial_state)
print("=" * 70)
print("FINAL GRAPH STATE")
print("=" * 70)

for key, value in final_product_state.items():
    print(f"{key}: {value}")

FINAL GRAPH STATE
product_name: Laptop
plan: Retrieve the price and category information for Laptop.
retrieved_data: {'product_id': 101, 'product': 'Laptop', 'price': 120000.0, 'category': 'Electronics'}
generated_answer: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.
formatted_answer: FINAL ANSWER: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.


In [ ]:
# Summarize the main Task 2 concepts demonstrated by the linear graph.
task2_summary = """
TASK 2 — LINEAR GRAPH

A linear LangGraph workflow executes nodes in a fixed sequence.

Each node receives the current shared state and returns updates to that state.

The Plan node decided what information was needed.

The Retrieve node calls lookup_product (via .invoke()), the same CSV-backed
tool built in Day 2, instead of duplicating that logic with a hardcoded dict.

The Generate node converted the retrieved data into an answer.

The Format node prepared the final response.

Unlike Task 1, this graph has no conditional routing and no cycle.
"""

print(task2_summary)


TASK 2 — LINEAR GRAPH

A linear LangGraph workflow executes nodes in a fixed sequence.

Each node receives the current shared state and returns updates to that state.

The Plan node decided what information was needed.

The Retrieve node calls lookup_product (via .invoke()), the same CSV-backed
tool built in Day 2, instead of duplicating that logic with a hardcoded dict.

The Generate node converted the retrieved data into an answer.

The Format node prepared the final response.

Unlike Task 1, this graph has no conditional routing and no cycle.



In [ ]:
# Extend the state schema with a quality score and a retry counter to support the self-correction loop.
class ProductWorkflowStateV2(TypedDict):
    product_name: str
    plan: str
    retrieved_data: dict
    generated_answer: str
    formatted_answer: str
    quality_score: float
    retry_count: int
    max_retries: int

print("ProductWorkflowStateV2 created successfully.")

ProductWorkflowStateV2 created successfully.


In [ ]:
# Reuse the existing plan and retrieve nodes as-is — no changes needed for Task 3.
plan_product_v2 = plan_product
retrieve_product_v2 = retrieve_product

print("Plan and Retrieve nodes reused from Task 2.")

Plan and Retrieve nodes reused from Task 2.


In [ ]:
# Generate node now also resets/produces an answer each pass, so the critique node has something fresh to check.
def generate_product_answer_v2(state: ProductWorkflowStateV2):
    data = state["retrieved_data"]
    if "error" in data:
        return {"generated_answer": data["error"]}
    answer = (
        f"The {data['product']} (ID: {data['product_id']}) costs {data['price']} PKR "
        f"and belongs to the {data['category']} category."
    )
    return {"generated_answer": answer}

print("Generate node (v2) created successfully.")

Generate node (v2) created successfully.


In [ ]:
# Critique node scores the answer — deliberately low on early passes so the retry loop actually runs and is visible.
def critique_product_answer(state: ProductWorkflowStateV2):
    retry_count = state["retry_count"]
    answer = state["generated_answer"]

    if "error" in answer.lower():
        score = 0.3
    elif retry_count < 1:
        score = 0.5  # force at least one retry pass to demonstrate the loop
    else:
        score = 0.9

    print(f"[CRITIQUE] Pass {retry_count + 1} — quality_score: {score}")

    return {"quality_score": score}

print("Critique node created successfully (forces at least one retry pass).")

Critique node created successfully (forces at least one retry pass).


In [ ]:
# Router checks quality against a threshold AND respects max_retries, so the loop can't run forever.
def critique_router_v2(state: ProductWorkflowStateV2):
    if state["quality_score"] >= 0.8:
        return "finish"
    if state["retry_count"] >= state["max_retries"]:
        print(f"[ROUTER] Max retries ({state['max_retries']}) reached — forcing finish.")
        return "finish"
    return "retry"

print("Critique router (with max-retries guard) created successfully.")

Critique router (with max-retries guard) created successfully.


In [ ]:
# Retry node bumps the counter and logs the loop-back before sending control back to generate.
def retry_product_answer(state: ProductWorkflowStateV2):
    new_count = state["retry_count"] + 1
    print(f"[RETRY] Looping back to generate — attempt {new_count} of {state['max_retries']}")
    return {"retry_count": new_count}

print("Retry node created successfully.")

Retry node created successfully.


In [ ]:
# Format node is unchanged from Task 2 — just wraps the final generated answer.
def format_product_answer_v2(state: ProductWorkflowStateV2):
    return {"formatted_answer": f"FINAL ANSWER: {state['generated_answer']}"}

print("Format node (v2) created successfully.")

Format node (v2) created successfully.


In [ ]:
# Build a new StateGraph for Task 3 using the extended state schema.
product_graph_builder_v2 = StateGraph(ProductWorkflowStateV2)

print("Task 3 StateGraph created successfully.")

Task 3 StateGraph created successfully.


In [ ]:
# Register all six nodes: plan, retrieve, generate, critique, retry, and format.
product_graph_builder_v2.add_node("plan", plan_product_v2)
product_graph_builder_v2.add_node("retrieve", retrieve_product_v2)
product_graph_builder_v2.add_node("generate", generate_product_answer_v2)
product_graph_builder_v2.add_node("critique", critique_product_answer)
product_graph_builder_v2.add_node("retry", retry_product_answer)
product_graph_builder_v2.add_node("format", format_product_answer_v2)

print("Task 3 nodes registered successfully.")

Task 3 nodes registered successfully.


In [ ]:
# Wire the fixed edges: plan -> retrieve -> generate -> critique, and format -> END.
product_graph_builder_v2.add_edge(START, "plan")
product_graph_builder_v2.add_edge("plan", "retrieve")
product_graph_builder_v2.add_edge("retrieve", "generate")
product_graph_builder_v2.add_edge("generate", "critique")
product_graph_builder_v2.add_edge("format", END)

print("Fixed edges added successfully.")

Fixed edges added successfully.


In [ ]:
# Add the conditional edge: critique routes to "format" (finish) or "retry" (loop back) based on quality/retries.
product_graph_builder_v2.add_conditional_edges(
    "critique",
    critique_router_v2,
    {
        "finish": "format",
        "retry": "retry"
    }
)

print("Conditional edge added successfully.")

Conditional edge added successfully.


In [ ]:
# Close the self-correction loop: retry sends control back to generate, not straight to critique.
product_graph_builder_v2.add_edge("retry", "generate")
print("Retry loop (retry -> generate) created successfully.")

Retry loop (retry -> generate) created successfully.


In [ ]:
# Compile the Task 3 graph so it becomes executable.
product_graph_v2 = product_graph_builder_v2.compile()
print("Task 3 graph compiled successfully.")

Task 3 graph compiled successfully.


In [ ]:
# Initial state includes retry_count starting at 0 and a max_retries cap of 2.
product_initial_state_v2: ProductWorkflowStateV2 = {
    "product_name": "Laptop",
    "plan": "",
    "retrieved_data": {},
    "generated_answer": "",
    "formatted_answer": "",
    "quality_score": 0.0,
    "retry_count": 0,
    "max_retries": 2
}
print("Initial state:")
print(product_initial_state_v2)

Initial state:
{'product_name': 'Laptop', 'plan': '', 'retrieved_data': {}, 'generated_answer': '', 'formatted_answer': '', 'quality_score': 0.0, 'retry_count': 0, 'max_retries': 2}


In [ ]:
# Stream execution so every node pass, including any retry loops, is visible in order.
print("=" * 70)
print("TASK 3 — SELF-CORRECTION GRAPH EXECUTION")
print("=" * 70)
for event in product_graph_v2.stream(product_initial_state_v2):
    for node_name, node_state in event.items():
        print("\n" + "=" * 70)
        print("NODE EXECUTED:", node_name)
        print("=" * 70)
        print("STATE UPDATE:")
        print(node_state)

TASK 3 — SELF-CORRECTION GRAPH EXECUTION

NODE EXECUTED: plan
STATE UPDATE:
{'plan': 'Retrieve the price and category information for Laptop.'}

NODE EXECUTED: retrieve
STATE UPDATE:
{'retrieved_data': {'product_id': 101, 'product': 'Laptop', 'price': 120000.0, 'category': 'Electronics'}}

NODE EXECUTED: generate
STATE UPDATE:
{'generated_answer': 'The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.'}
[CRITIQUE] Pass 1 — quality_score: 0.9

NODE EXECUTED: critique
STATE UPDATE:
{'quality_score': 0.9}

NODE EXECUTED: format
STATE UPDATE:
{'formatted_answer': 'FINAL ANSWER: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.'}


In [ ]:
# Invoke once more and print the final accumulated state to confirm the loop terminated correctly.
final_state_v2 = product_graph_v2.invoke(product_initial_state_v2)
print("=" * 70)
print("FINAL GRAPH STATE")
print("=" * 70)
for key, value in final_state_v2.items():
    print(f"{key}: {value}")

[CRITIQUE] Pass 1 — quality_score: 0.9
FINAL GRAPH STATE
product_name: Laptop
plan: Retrieve the price and category information for Laptop.
retrieved_data: {'product_id': 101, 'product': 'Laptop', 'price': 120000.0, 'category': 'Electronics'}
generated_answer: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.
formatted_answer: FINAL ANSWER: The Laptop (ID: 101) costs 120000.0 PKR and belongs to the Electronics category.
quality_score: 0.9
retry_count: 0
max_retries: 2


In [ ]:
# Explain why this loop-back pattern is natural in LangGraph but awkward in a plain AgentExecutor.
task3_explanation = """
TASK 3 — WHY THIS IS HARD IN AgentExecutor, EASY IN LANGGRAPH

AgentExecutor runs one linear loop (model -> tool call -> observation -> repeat) with no
concept of named steps or "go back to step X," so faking a critique-and-retry cycle means
smuggling that logic into the prompt or wrapping the executor in your own outer while-loop.
LangGraph instead treats the workflow as an explicit graph, where quality_score and
retry_count are just state fields and add_conditional_edges lets any node route back to
an earlier one — making the loop a first-class part of the graph definition, not a workaround.
"""

print(task3_explanation)


TASK 3 — WHY THIS IS HARD IN AgentExecutor, EASY IN LANGGRAPH

AgentExecutor runs one linear loop (model -> tool call -> observation -> repeat) with no
concept of named steps or "go back to step X," so faking a critique-and-retry cycle means
smuggling that logic into the prompt or wrapping the executor in your own outer while-loop.
LangGraph instead treats the workflow as an explicit graph, where quality_score and
retry_count are just state fields and add_conditional_edges lets any node route back to
an earlier one — making the loop a first-class part of the graph definition, not a workaround.



START
  ↓
prepare_purchase
  ↓
human_approval  ← interrupt happens here
  ↓
   ┌───────────────┐
   │               │
APPROVED        REJECTED
   ↓               ↓
purchase        cancel
   ↓               ↓
  END             END

In [ ]:
# Import the LangGraph interrupt, resume command, and checkpointing components for human approval.
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
print("Task 4 dependencies imported successfully.")

Task 4 dependencies imported successfully.


In [ ]:
# Define a dedicated state schema for the human-approval purchase workflow.
class PurchaseApprovalState(TypedDict):
    product_name: str
    price: float
    approval_status: str
    action_result: str
print("PurchaseApprovalState created successfully.")

PurchaseApprovalState created successfully.


In [ ]:
# Prepare the purchase information before asking the human for permission.
def prepare_purchase(state: PurchaseApprovalState):
    print("[PREPARE] Preparing purchase request...")
    return {
        "approval_status": "pending"
    }

print("Prepare-purchase node created successfully.")

Prepare-purchase node created successfully.


In [ ]:
# Pause the graph and request human approval before performing the risky purchase action.
def request_human_approval(state: PurchaseApprovalState):
    print("[HUMAN APPROVAL] Waiting for human decision...")
    decision = interrupt({
        "message": "Human approval required before purchasing.",
        "product": state["product_name"],
        "price": state["price"],
        "options": ["approve", "reject"]
    })
    return {
        "approval_status": decision
    }
print("Human-approval interrupt node created successfully.")

Human-approval interrupt node created successfully.


In [ ]:
# Execute the simulated risky purchase only after the human has approved it.
def execute_purchase(state: PurchaseApprovalState):
    print("[PURCHASE] Human approved the action.")
    return {
        "action_result": (
            f"Purchase completed successfully for "
            f"{state['product_name']} at {state['price']} PKR."
        )
    }

print("Purchase execution node created successfully.")

Purchase execution node created successfully.


In [ ]:
# Cancel the risky action when the human rejects the request.
def cancel_purchase(state: PurchaseApprovalState):
    print("[CANCEL] Human rejected the action.")
    return {
        "action_result": (
            f"Purchase cancelled for {state['product_name']}."
        )
    }

print("Cancellation node created successfully.")

Cancellation node created successfully.


In [ ]:
# Route the workflow to purchase or cancellation based on the human's decision.
def approval_router(state: PurchaseApprovalState):
    if state["approval_status"].lower() == "approve":
        return "approved"

    return "rejected"

print("Approval router created successfully.")

Approval router created successfully.


In [ ]:
# Create a completely separate StateGraph for the human-in-the-loop workflow.

purchase_graph_builder = StateGraph(PurchaseApprovalState)
print("Task 4 StateGraph created successfully.")

Task 4 StateGraph created successfully.


In [ ]:
# Register the preparation, approval, purchase, and cancellation nodes.
purchase_graph_builder.add_node("prepare", prepare_purchase)
purchase_graph_builder.add_node("human_approval", request_human_approval)
purchase_graph_builder.add_node("purchase", execute_purchase)
purchase_graph_builder.add_node("cancel", cancel_purchase)
print("Task 4 nodes registered successfully.")

Task 4 nodes registered successfully.


In [ ]:
# Connect the initial preparation node to the human approval checkpoint.
purchase_graph_builder.add_edge(START, "prepare")
purchase_graph_builder.add_edge("prepare", "human_approval")

print("Initial Task 4 edges added successfully.")

Initial Task 4 edges added successfully.


In [ ]:
# Route the workflow to either purchase or cancellation after human approval.
purchase_graph_builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "approved": "purchase",
        "rejected": "cancel"
    }
)
print("Human approval routing added successfully.")

Human approval routing added successfully.


In [ ]:
# Finish the workflow after either the approved purchase or rejected cancellation.

purchase_graph_builder.add_edge("purchase", END)
purchase_graph_builder.add_edge("cancel", END)

print("Task 4 ending edges added successfully.")

Task 4 ending edges added successfully.


In [ ]:
# Create an in-memory checkpointer so LangGraph can persist the paused graph state.
memory_saver = MemorySaver()
print("MemorySaver checkpointer created successfully.")

MemorySaver checkpointer created successfully.


In [ ]:
# Compile the graph with the checkpointer so interrupted executions can be resumed.
purchase_graph = purchase_graph_builder.compile(
    checkpointer=memory_saver
)

print("Task 4 graph compiled with checkpointing successfully.")

Task 4 graph compiled with checkpointing successfully.


In [ ]:
# Display the intended Task 4 workflow structure before executing it.

print("""
TASK 4 — HUMAN-IN-THE-LOOP GRAPH

START
  |
  v
PREPARE
  |
  v
HUMAN APPROVAL
  |
  +------ approve ------> PURCHASE ------> END
  |
  +------ reject -------> CANCEL  -------> END
""")


TASK 4 — HUMAN-IN-THE-LOOP GRAPH

START
  |
  v
PREPARE
  |
  v
HUMAN APPROVAL
  |
  +------ approve ------> PURCHASE ------> END
  |
  +------ reject -------> CANCEL  -------> END



In [ ]:
# Create the initial state representing a purchase that requires human approval.
purchase_initial_state: PurchaseApprovalState = {
    "product_name": "Laptop",
    "price": 120000.0,
    "approval_status": "",
    "action_result": ""
}

print("Initial purchase state:")
print(purchase_initial_state)

Initial purchase state:
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': '', 'action_result': ''}


In [ ]:
# Create a thread identifier so LangGraph knows which paused execution should be resumed.
config = {
    "configurable": {
        "thread_id": "purchase-request-001"
    }
}

print("Thread configuration created.")

Thread configuration created.


In [ ]:
# Run the graph until it reaches the human approval interrupt.
paused_result = purchase_graph.invoke(
    purchase_initial_state,
    config
)

print("Graph execution paused for human approval.")
print(paused_result)

[PREPARE] Preparing purchase request...
[HUMAN APPROVAL] Waiting for human decision...
Graph execution paused for human approval.
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': '', '__interrupt__': [Interrupt(value={'message': 'Human approval required before purchasing.', 'product': 'Laptop', 'price': 120000.0, 'options': ['approve', 'reject']}, id='0a2d329bcd204458e4713497ca472b05')]}


In [ ]:
# Inspect the current graph state to confirm that execution is waiting for human input.
current_state = purchase_graph.get_state(config)
print("Current graph state:")
print(current_state)
print("\nInterrupt information:")
print(current_state.interrupts)

Current graph state:
StateSnapshot(values={'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': ''}, next=('human_approval',), config={'configurable': {'thread_id': 'purchase-request-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19632e-f73d-68d6-8001-084d78d7bd66'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-12T09:48:13.507322+00:00', parent_config={'configurable': {'thread_id': 'purchase-request-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19632e-f723-6887-8000-a4bdc22cedb1'}}, tasks=(PregelTask(id='b5beb4b8-e744-f3eb-ca9e-5c3f8c0cd909', name='human_approval', path=('__pregel_pull', 'human_approval'), error=None, interrupts=(Interrupt(value={'message': 'Human approval required before purchasing.', 'product': 'Laptop', 'price': 120000.0, 'options': ['approve', 'reject']}, id='0a2d329bcd204458e4713497ca472b05'),), state=None, result=None),), interrupts=(Interrupt(value={'message': 'Human approval required before 

In [ ]:
# Resume the paused graph by providing the simulated human approval decision.

approved_result = purchase_graph.invoke(
    Command(resume="approve"),
    config
)
print("Graph resumed after human approval.")
print(approved_result)

[HUMAN APPROVAL] Waiting for human decision...
[PURCHASE] Human approved the action.
Graph resumed after human approval.
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'approve', 'action_result': 'Purchase completed successfully for Laptop at 120000.0 PKR.'}


In [ ]:
# Display the final state after the approved purchase workflow completes.
print("=" * 70)
print("APPROVED WORKFLOW RESULT")
print("=" * 70)

for key, value in approved_result.items():
    print(f"{key}: {value}")

APPROVED WORKFLOW RESULT
product_name: Laptop
price: 120000.0
approval_status: approve
action_result: Purchase completed successfully for Laptop at 120000.0 PKR.


In [ ]:
# Create a separate purchase request so the rejection path has its own checkpointed execution.

rejection_config = {
    "configurable": {
        "thread_id": "purchase-request-002"
    }
}
rejection_initial_state: PurchaseApprovalState = {
    "product_name": "Monitor",
    "price": 35000.0,
    "approval_status": "",
    "action_result": ""
}
print("Second purchase request created.")

Second purchase request created.


In [ ]:
# Start the second workflow and pause it at the human approval checkpoint.
purchase_graph.invoke(
    rejection_initial_state,
    rejection_config
)

print("Second workflow paused for human approval.")

[PREPARE] Preparing purchase request...
[HUMAN APPROVAL] Waiting for human decision...
Second workflow paused for human approval.


In [ ]:
# Resume the second workflow with a simulated human rejection.
rejected_result = purchase_graph.invoke(
    Command(resume="reject"),
    rejection_config
)

print("Graph resumed after human rejection.")

[HUMAN APPROVAL] Waiting for human decision...
[CANCEL] Human rejected the action.
Graph resumed after human rejection.


In [ ]:
# Display the final state after the human rejected the risky action.
print("=" * 70)
print("REJECTED WORKFLOW RESULT")
print("=" * 70)
for key, value in rejected_result.items():
    print(f"{key}: {value}")

REJECTED WORKFLOW RESULT
product_name: Monitor
price: 35000.0
approval_status: reject
action_result: Purchase cancelled for Monitor.


In [ ]:
# Compare the approved and rejected workflows to verify both human-in-the-loop paths.
print("APPROVED PATH:")
print(approved_result["action_result"])
print("\nREJECTED PATH:")
print(rejected_result["action_result"])

APPROVED PATH:
Purchase completed successfully for Laptop at 120000.0 PKR.

REJECTED PATH:
Purchase cancelled for Monitor.


In [ ]:
# Discuss when human-in-the-loop is actually necessary vs. when full autonomy is fine.
task4_discussion = """
TASK 4 — WHEN TO REQUIRE HUMAN APPROVAL

Human-in-the-loop is worth the added latency when an action is costly, hard to reverse,
or touches real money/identity/external systems — like an actual purchase or emailing a
real customer — since a wrong autonomous decision there is expensive to undo. Full
autonomy is fine for cheap, reversible, low-stakes actions like a calculator call or a
read-only lookup, where a wrong answer just means the user asks again. The practical rule:
gate only the specific risky node on approval, not the whole workflow, or the human becomes
a bottleneck for decisions that were never actually dangerous.
"""

print(task4_discussion)


TASK 4 — WHEN TO REQUIRE HUMAN APPROVAL

Human-in-the-loop is worth the added latency when an action is costly, hard to reverse,
or touches real money/identity/external systems — like an actual purchase or emailing a
real customer — since a wrong autonomous decision there is expensive to undo. Full
autonomy is fine for cheap, reversible, low-stakes actions like a calculator call or a
read-only lookup, where a wrong answer just means the user asks again. The practical rule:
gate only the specific risky node on approval, not the whole workflow, or the human becomes
a bottleneck for decisions that were never actually dangerous.



In [ ]:
# Import the checkpointing and command components needed to persist and resume graph executions.
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

print("Task 5 dependencies imported successfully.")

Task 5 dependencies imported successfully.


In [ ]:
# Create an in-memory checkpointer that stores graph state for each thread.
task5_memory = MemorySaver()
print("Task 5 MemorySaver created successfully.")

Task 5 MemorySaver created successfully.


In [ ]:
# Rebuild the Task 4 purchase graph with the new Task 5 checkpointer.
task5_graph_builder = StateGraph(PurchaseApprovalState)
task5_graph_builder.add_node("prepare", prepare_purchase)
task5_graph_builder.add_node("human_approval", request_human_approval)
task5_graph_builder.add_node("purchase", execute_purchase)
task5_graph_builder.add_node("cancel", cancel_purchase)

task5_graph_builder.add_edge(START, "prepare")
task5_graph_builder.add_edge("prepare", "human_approval")

task5_graph_builder.add_conditional_edges(
    "human_approval",
    approval_router,
    {
        "approved": "purchase",
        "rejected": "cancel"
    }
)

task5_graph_builder.add_edge("purchase", END)
task5_graph_builder.add_edge("cancel", END)

print("Task 5 graph structure created successfully.")

Task 5 graph structure created successfully.


In [ ]:
# Compile the graph with MemorySaver so every execution checkpoint is stored.
task5_graph = task5_graph_builder.compile(
    checkpointer=task5_memory
)

print("Task 5 graph compiled with persistence successfully.")

Task 5 graph compiled with persistence successfully.


In [ ]:
# Create a unique thread ID that identifies this particular workflow conversation.
task5_config = {
    "configurable": {
        "thread_id": "task5-purchase-001"
    }
}

print("Persistent thread created:", task5_config)

Persistent thread created: {'configurable': {'thread_id': 'task5-purchase-001'}}


In [ ]:
# Create the initial purchase request that will be saved and later resumed.
task5_initial_state: PurchaseApprovalState = {
    "product_name": "Laptop",
    "price": 120000.0,
    "approval_status": "",
    "action_result": ""
}

print("Initial Task 5 state:")
print(task5_initial_state)

Initial Task 5 state:
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': '', 'action_result': ''}


In [ ]:
# Start the workflow and allow it to pause when human approval is required.
task5_paused = task5_graph.invoke(
    task5_initial_state,
    task5_config
)

print("Workflow reached the human approval checkpoint.")
print(task5_paused)

[PREPARE] Preparing purchase request...
[HUMAN APPROVAL] Waiting for human decision...
Workflow reached the human approval checkpoint.
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': '', '__interrupt__': [Interrupt(value={'message': 'Human approval required before purchasing.', 'product': 'Laptop', 'price': 120000.0, 'options': ['approve', 'reject']}, id='bd4712f453cb2de263aed4ada5d24c97')]}


In [ ]:
# Read the persisted checkpoint to verify that LangGraph saved the workflow state.
task5_saved_state = task5_graph.get_state(task5_config)

print("=" * 70)
print("SAVED CHECKPOINT")
print("=" * 70)
print(task5_saved_state)

SAVED CHECKPOINT
StateSnapshot(values={'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': ''}, next=('human_approval',), config={'configurable': {'thread_id': 'task5-purchase-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19635e-7eb8-6758-8001-7301418a4fba'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-12T10:09:29.360138+00:00', parent_config={'configurable': {'thread_id': 'task5-purchase-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19635e-7ea6-6857-8000-3bb6fd7839fb'}}, tasks=(PregelTask(id='f9e7136b-ba5d-b737-ae5e-91c94f44d635', name='human_approval', path=('__pregel_pull', 'human_approval'), error=None, interrupts=(Interrupt(value={'message': 'Human approval required before purchasing.', 'product': 'Laptop', 'price': 120000.0, 'options': ['approve', 'reject']}, id='bd4712f453cb2de263aed4ada5d24c97'),), state=None, result=None),), interrupts=(Interrupt(value={'message': 'Human approval required before purchasi

In [ ]:
# Display the important state values stored inside the checkpoint.
print("Product:", task5_saved_state.values["product_name"])
print("Price:", task5_saved_state.values["price"])
print("Approval status:", task5_saved_state.values["approval_status"])
print("Action result:", task5_saved_state.values["action_result"])

Product: Laptop
Price: 120000.0
Approval status: pending
Action result: 


In [ ]:
# Resume the exact same paused workflow using its existing thread ID and an approval decision.
task5_approved = task5_graph.invoke(
    Command(resume="approve"),
    task5_config
)

print("Workflow successfully resumed after human approval.")
print(task5_approved)

[HUMAN APPROVAL] Waiting for human decision...
[PURCHASE] Human approved the action.
Workflow successfully resumed after human approval.
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'approve', 'action_result': 'Purchase completed successfully for Laptop at 120000.0 PKR.'}


In [ ]:
# Verify that the resumed workflow completed the risky action after approval.
print("=" * 70)
print("RESUMED WORKFLOW RESULT")
print("=" * 70)

for key, value in task5_approved.items():
    print(f"{key}: {value}")

RESUMED WORKFLOW RESULT
product_name: Laptop
price: 120000.0
approval_status: approve
action_result: Purchase completed successfully for Laptop at 120000.0 PKR.


In [ ]:
# Retrieve all checkpoints belonging to this workflow thread for debugging and replay analysis.
history = list(task5_graph.get_state_history(task5_config))

print("Number of checkpoints:", len(history))

Number of checkpoints: 5


In [ ]:
# Print each historical checkpoint so we can inspect how the workflow progressed over time.
for index, checkpoint in enumerate(reversed(history), start=1):
    print("\n" + "=" * 70)
    print(f"CHECKPOINT {index}")
    print("=" * 70)
    print("Values:", checkpoint.values)
    print("Next nodes:", checkpoint.next)


CHECKPOINT 1
Values: {}
Next nodes: ('__start__',)

CHECKPOINT 2
Values: {'product_name': 'Laptop', 'price': 120000.0, 'approval_status': '', 'action_result': ''}
Next nodes: ('prepare',)

CHECKPOINT 3
Values: {'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': ''}
Next nodes: ('human_approval',)

CHECKPOINT 4
Values: {'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'approve', 'action_result': ''}
Next nodes: ('purchase',)

CHECKPOINT 5
Values: {'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'approve', 'action_result': 'Purchase completed successfully for Laptop at 120000.0 PKR.'}
Next nodes: ()


In [ ]:
# Search the history for a checkpoint where the workflow was waiting for human approval.
approval_checkpoint = None

for checkpoint in history:
    if checkpoint.values.get("approval_status") == "pending":
        approval_checkpoint = checkpoint
        break

print("Approval checkpoint found:", approval_checkpoint is not None)

Approval checkpoint found: True


In [ ]:
# Display the historical approval checkpoint to understand exactly what the agent knew at that moment.
if approval_checkpoint:
    print("=" * 70)
    print("HISTORICAL APPROVAL STATE")
    print("=" * 70)

    for key, value in approval_checkpoint.values.items():
        print(f"{key}: {value}")
else:
    print("No pending approval checkpoint found.")

HISTORICAL APPROVAL STATE
product_name: Laptop
price: 120000.0
approval_status: pending
action_result: 


In [ ]:
# Display the checkpoint configuration so we can identify the historical execution point.
if approval_checkpoint:
    print("Historical checkpoint configuration:")
    print(approval_checkpoint.config)

Historical checkpoint configuration:
{'configurable': {'thread_id': 'task5-purchase-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19635e-7eb8-6758-8001-7301418a4fba'}}


In [ ]:
# Read the graph state represented by the selected historical checkpoint.
if approval_checkpoint:
    historical_state = task5_graph.get_state(
        approval_checkpoint.config
    )

    print("=" * 70)
    print("HISTORICAL STATE SNAPSHOT")
    print("=" * 70)
    print(historical_state)

HISTORICAL STATE SNAPSHOT
StateSnapshot(values={'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'pending', 'action_result': ''}, next=('human_approval',), config={'configurable': {'thread_id': 'task5-purchase-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19635e-7eb8-6758-8001-7301418a4fba'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-12T10:09:29.360138+00:00', parent_config={'configurable': {'thread_id': 'task5-purchase-001', 'checkpoint_ns': '', 'checkpoint_id': '1f19635e-7ea6-6857-8000-3bb6fd7839fb'}}, tasks=(PregelTask(id='f9e7136b-ba5d-b737-ae5e-91c94f44d635', name='human_approval', path=('__pregel_pull', 'human_approval'), error=None, interrupts=(Interrupt(value={'message': 'Human approval required before purchasing.', 'product': 'Laptop', 'price': 120000.0, 'options': ['approve', 'reject']}, id='bd4712f453cb2de263aed4ada5d24c97'),), state=None, result={'approval_status': 'approve'}),), interrupts=(Interrupt(value={'message': 'Huma

In [ ]:
# Create a small helper that converts a LangGraph checkpoint into an easy-to-read debugging snapshot.
def print_state_snapshot(label, checkpoint):
    print("\n" + "=" * 70)
    print(label)
    print("=" * 70)

    for key, value in checkpoint.values.items():
        print(f"{key}: {value}")

    print("Next nodes:", checkpoint.next)

In [ ]:
# Capture and display the final checkpoint as a simple debugging snapshot.
current_task5_state = task5_graph.get_state(task5_config)
print_state_snapshot(
    "CURRENT GRAPH SNAPSHOT",
    current_task5_state
)


CURRENT GRAPH SNAPSHOT
product_name: Laptop
price: 120000.0
approval_status: approve
action_result: Purchase completed successfully for Laptop at 120000.0 PKR.
Next nodes: ()


In [ ]:
# Access the same thread ID again to demonstrate that the saved workflow state remains available.
same_thread_state = task5_graph.get_state(task5_config)
print("State recovered from the same thread:")
print(same_thread_state.values)

State recovered from the same thread:
{'product_name': 'Laptop', 'price': 120000.0, 'approval_status': 'approve', 'action_result': 'Purchase completed successfully for Laptop at 120000.0 PKR.'}


In [ ]:
# Compare AgentExecutor and LangGraph — rendered as an actual Markdown table.
from IPython.display import display, Markdown

comparison = """
### AGENTEXECUTOR VS LANGGRAPH

| Aspect | AgentExecutor | LangGraph |
|---|---|---|
| Core model | model -> tool -> observation -> answer | explicit graph of nodes and edges |
| Branching | not supported cleanly | conditional edges, native support |
| Cycles / retries | requires manual outer while-loop | native, via edges looping back to earlier nodes |
| State | implicit, in-memory only | explicit, typed State object |
| Human-in-the-loop | not built in | native, via interrupt() + checkpointer |
| Persistence across runs | not built in | native, via checkpointer (e.g. MemorySaver) |
| Debugging / replay | limited | get_state_history() gives full step-by-step trace |
| Best for | simple tool-using chatbots | research, approvals, financial actions, multi-step agents |
| Setup complexity | lower, faster to write | higher upfront, pays off as workflow grows |

**Practical rule:** Simple agent loop → AgentExecutor. Complex stateful workflow → LangGraph.
"""

display(Markdown(comparison))


### AGENTEXECUTOR VS LANGGRAPH

| Aspect | AgentExecutor | LangGraph |
|---|---|---|
| Core model | model -> tool -> observation -> answer | explicit graph of nodes and edges |
| Branching | not supported cleanly | conditional edges, native support |
| Cycles / retries | requires manual outer while-loop | native, via edges looping back to earlier nodes |
| State | implicit, in-memory only | explicit, typed State object |
| Human-in-the-loop | not built in | native, via interrupt() + checkpointer |
| Persistence across runs | not built in | native, via checkpointer (e.g. MemorySaver) |
| Debugging / replay | limited | get_state_history() gives full step-by-step trace |
| Best for | simple tool-using chatbots | research, approvals, financial actions, multi-step agents |
| Setup complexity | lower, faster to write | higher upfront, pays off as workflow grows |

**Practical rule:** Simple agent loop → AgentExecutor. Complex stateful workflow → LangGraph.


```mermaid
flowchart TD
    START([START]) --> PLAN[Plan]
    PLAN --> RETRIEVE[Retrieve Data]
    RETRIEVE --> GENERATE[Generate Answer]
    GENERATE --> CRITIQUE[Critique]

    CRITIQUE -->|Quality >= 0.8| PREPARE[Prepare Risky Action]
    CRITIQUE -->|Quality < 0.8| RETRY[Retry / Revise]

    RETRY --> GENERATE

    PREPARE --> APPROVAL{Human Approval}
    APPROVAL -->|Approve| ACTION[Execute Risky Action]
    APPROVAL -->|Reject| CANCEL[Cancel Action]

    ACTION --> END([END])
    CANCEL --> END

    CHECKPOINT[(MemorySaver Checkpoint)] -. persists .-> PREPARE
    CHECKPOINT -. resume by thread_id .-> APPROVAL

    HISTORY[(State History)] -. debugging .-> CHECKPOINT
```

In [ ]:
# Summarize the persistence and debugging capabilities demonstrated in Task 5.
task5_summary = """
TASK 5 — PERSISTENCE & DEBUGGING

1. MemorySaver stores graph checkpoints associated with a thread_id.

2. The same thread_id allows a paused workflow to be resumed later.

3. Command(resume=...) supplies the human decision required by an interrupt.

4. get_state() allows us to inspect the current persisted state.

5. get_state_history() allows us to inspect previous checkpoints.

6. Historical checkpoints make debugging easier because we can inspect
   what state existed at different points in the workflow.

7. AgentExecutor is preferable for simpler agent loops.

8. LangGraph is preferable for complex, stateful, branching, cyclical,
   interruptible, and persistent agent workflows.
"""

print(task5_summary)


TASK 5 — PERSISTENCE & DEBUGGING

1. MemorySaver stores graph checkpoints associated with a thread_id.

2. The same thread_id allows a paused workflow to be resumed later.

3. Command(resume=...) supplies the human decision required by an interrupt.

4. get_state() allows us to inspect the current persisted state.

5. get_state_history() allows us to inspect previous checkpoints.

6. Historical checkpoints make debugging easier because we can inspect
   what state existed at different points in the workflow.

7. AgentExecutor is preferable for simpler agent loops.

8. LangGraph is preferable for complex, stateful, branching, cyclical,
   interruptible, and persistent agent workflows.

